# Interpolação polinomial: atividade resolvida

**Aluno:** Pedro Henrique Moreira Oliveira  
**Matrícula:** 188198  
**Disciplina:** Cálculo Numérico

Este notebook apresenta as respostas, cálculos e conclusões da atividade sobre interpolação polinomial.

## 1. Ideia central

Interpolar significa estimar um valor **dentro** do intervalo coberto pelos nós conhecidos. Extrapolar significa estimar um valor **fora** desse intervalo.

Seria útil interpolar para estimar a temperatura às 14h30 quando existem medições às 14h e às 15h. A estimativa pode ser pouco confiável se houver ruído, poucos pontos, mudanças bruscas no fenômeno ou medições imprecisas.

## 2. Nós e grau

Se acrescentarmos um novo ponto, o polinômio continuará passando pelos pontos antigos e também pelo novo ponto. Entretanto, seu grau e seu formato podem mudar. Com nós distintos, existe um único polinômio de grau no máximo igual ao número de nós menos um.

## 3. Primeiro conjunto de pontos

Os pontos `(-2, 9)`, `(-1, 4)`, `(0, 1)` e `(1, 0)` apresentam uma tendência parabólica. Eles pertencem à função $P(x)=(x-1)^2=x^2-2x+1$. Portanto, a curva não precisa ser reta nem cúbica; o polinômio interpolador tem grau 2.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.array([-2, -1, 0, 1], dtype=float)
y = np.array([9, 4, 1, 0], dtype=float)
polinomio = np.polynomial.Polynomial.fit(x, y, deg=len(x)-1).convert()

print('Polinômio na forma usual:')
print(polinomio)
print('Coeficientes:', polinomio.coef)

xx = np.linspace(-2.5, 1.5, 400)
plt.figure(figsize=(8, 4))
plt.plot(xx, polinomio(xx), label='Polinômio interpolador')
plt.scatter(x, y, color='crimson', zorder=3, label='Nós')
plt.axvspan(-2, 1, color='green', alpha=0.08, label='Intervalo de interpolação')
plt.xlabel('x'); plt.ylabel('P(x)'); plt.grid(True); plt.legend(); plt.show()

### Interpretação

O resultado numérico é aproximadamente $P(x)=1-2x+x^2-8{,}18\times10^{-16}x^3$. O coeficiente cúbico é praticamente zero, portanto $P(x)=x^2-2x+1$.

As diferenças entre $P(x_i)$ e $y_i$ são da ordem de $10^{-15}$, apenas erros de arredondamento. Isso confirma que o polinômio reproduz os dados nos nós, mas não garante que ele represente corretamente valores entre os nós ou fora do intervalo.

In [ ]:
consultas = np.array([-1.5, -0.5, 0.5, 0.9, 2.0])
for q in consultas:
    tipo = 'interpolação' if x.min() <= q <= x.max() else 'extrapolação'
    print(f'P({q:4.1f}) = {polinomio(q): .6f} ({tipo})')

### Tabela de valores

| $x$ | $P(x)$ | Tipo |
|---:|---:|---|
| -1,5 | 6,25 | Interpolação |
| -0,5 | 2,25 | Interpolação |
| 0,5 | 0,25 | Interpolação |
| 0,9 | 0,01 | Interpolação |
| 2,0 | 1,00 | Extrapolação |

A curva passa pelos pontos. Dentro de $[-2,1]$, o comportamento é sustentado pelos dados; fora dele, trata-se de extrapolação e é preciso ter cautela.

## 4. Comparação com outras ferramentas

**GeoGebra:** usar `Polynomial({(-2,9),(-1,4),(0,1),(1,0)})`.

**WolframAlpha:** informar os pontos `(-2,9), (-1,4), (0,1), (1,0)` e solicitar o polinômio interpolador.

As duas ferramentas retornam $P(x)=x^2-2x+1$, com $P(-1,5)=6,25$ e $P(0,5)=0,25$. Os resultados concordam porque foi usada interpolação polinomial exata, e não regressão ou spline.

Links: [manual do comando Polynomial do GeoGebra](https://geogebra.github.io/docs/manual/en/commands/Polynomial/) e [calculadora de polinômio interpolador do WolframAlpha](https://www.wolframalpha.com/calculators/mathematics-statistics-regression-interpolating-polynomial-calculator).

## 5. Segundo conjunto: dados de medição

Para os pontos `(0, 1,2)`, `(1, 2,0)`, `(2, 2,7)` e `(3, 4,1)`, o polinômio interpolador é

$$P_B(x)=1{,}2+1{,}1166667x-0{,}45x^2+0{,}1333333x^3.$$

Como $x=1,5$ está entre 0 e 3, é interpolação: $P_B(1,5)=2,3125$.

In [ ]:
xB = np.array([0, 1, 2, 3], dtype=float)
yB = np.array([1.2, 2.0, 2.7, 4.1], dtype=float)
pB = np.polynomial.Polynomial.fit(xB, yB, deg=3).convert()
print(pB)
print(f'Estimativa em x=1.5: {pB(1.5):.6f}')
xb_plot = np.linspace(0, 3, 300)
plt.figure(figsize=(8, 4))
plt.plot(xb_plot, pB(xb_plot), label='Interpolante')
plt.scatter(xB, yB, color='crimson', zorder=3, label='Medições')
plt.xlabel('Tempo x'); plt.ylabel('Medida y'); plt.grid(True); plt.legend(); plt.show()

## 6. Alteração de uma medição

O valor medido em $x=2$ foi alterado de $2,7$ para $3,2$, um aumento de $0,5$. A estimativa em $x=1,5$ mudou de $2,3125$ para $2,59375$, uma variação de $0,28125$.

A curva não muda somente perto do ponto alterado: ela muda em todo o intervalo, embora continue passando pelos nós. Por isso, pequenas incertezas de medição podem afetar significativamente as estimativas.

In [ ]:
yB_alterado = yB.copy()
yB_alterado[2] += 0.5
pB_alterado = np.polynomial.Polynomial.fit(xB, yB_alterado, deg=3).convert()
print('Estimativa original:', pB(1.5))
print('Estimativa alterada:', pB_alterado(1.5))
print('Variação:', pB_alterado(1.5) - pB(1.5))
plt.figure(figsize=(8, 4))
plt.plot(xb_plot, pB(xb_plot), label='Original')
plt.plot(xb_plot, pB_alterado(xb_plot), '--', label='Com y(2) aumentado em 0,5')
plt.scatter(xB, yB, color='crimson', label='Dados originais')
plt.scatter(xB, yB_alterado, marker='x', s=80, color='black', label='Dados alterados')
plt.xlabel('x'); plt.ylabel('y'); plt.grid(True); plt.legend(); plt.show()

## 7. Muitos nós e fenômeno de Runge

Ao interpolar a função $f(x)=1/(1+25x^2)$ com nós igualmente espaçados, todas as curvas passam pelos nós, mas podem oscilar perto das extremidades. Com 5, 9 e 15 nós, os erros máximos aproximados são, respectivamente, $0,438$, $1,045$ e $7,195$.

Isso mostra que aumentar o número de pontos e exigir ajuste exato não garante uma curva melhor. Para muitos dados, eu investigaria splines, interpolação por partes ou nós de Chebyshev.

In [ ]:
def f(t):
    return 1 / (1 + 25*t**2)

t_plot = np.linspace(-1, 1, 1000)
plt.figure(figsize=(9, 5))
plt.plot(t_plot, f(t_plot), color='black', linewidth=2, label='Função de referência')
for n in [5, 9, 15]:
    tn = np.linspace(-1, 1, n)
    pn = np.polynomial.Polynomial.fit(tn, f(tn), deg=n-1).convert()
    plt.plot(t_plot, pn(t_plot), label=f'Interpolante com {n} nós')
    plt.scatter(tn, f(tn), s=18)
plt.ylim(-0.5, 1.2); plt.grid(True); plt.legend()
plt.xlabel('x'); plt.ylabel('y'); plt.title('Fenômeno de Runge')
plt.show()

## 8. Interpolação versus regressão

Na interpolação, a curva passa por todos os dados. Na regressão, o modelo procura representar a tendência geral e não precisa passar por cada observação. Assim, o fato de a curva do conjunto B passar exatamente pelas medições não prova que elas sejam exatas ou sem ruído.

## 9. Síntese

Interpolar significa estimar um valor dentro do intervalo coberto pelos nós conhecidos.
Extrapolar significa estimar fora desse intervalo, sendo geralmente mais arriscado.
O polinômio interpolador passa exatamente pelos pontos fornecidos.
Foram comparados NumPy, GeoGebra e WolframAlpha, que produziram resultados concordantes.
No conjunto B, alterar uma medição mudou o valor estimado em $x=1,5$.
A interpolação polinomial pode apresentar oscilações quando possui grau elevado.
O fenômeno de Runge mostrou que mais pontos não garantem uma curva melhor.
Eu usaria interpolação entre medições confiáveis e evitaria extrapolações longas.